# 00 - Environment Understanding

**Goal:** Understand what SoccerTwos gives the project before training anything.

**What you will learn:** Training vs live-match spaces, sparse rewards, info fields, and reward shaping at a project-specific level.

**Inputs:** An installed `soccertwos` environment and the organized package.

**Outputs:** Environment sanity checks, compact rollout logs, and a reward-shaping signal example.

**Success criteria:** The environment launches headlessly and you can explain observation/action/reward flow.

In [1]:
from pathlib import Path
import importlib
import os
import sys

PROJECT_MARKER = Path("soccer_twos_project") / "notebook_tools.py"


def _running_in_colab():
    if "google.colab" in sys.modules:
        return True
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _candidate_project_roots():
    seen = set()

    def add(path):
        path = Path(path).expanduser()
        key = str(path)
        if key not in seen:
            seen.add(key)
            yield path

    for env_name in ("SOCCER_TWOS_PROJECT_ROOT", "PROJECT_ROOT"):
        value = os.environ.get(env_name)
        if value:
            yield from add(value)

    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        yield from add(base)
        yield from add(base / "soccer-twos-starter")
        yield from add(base / "project" / "soccer-twos-starter")

    if sys.platform == "darwin":
        yield from add(
            Path.home()
            / "all_data"
            / "Georgia Tech"
            / "Course Content"
            / "CS 8803- DRL"
            / "project"
            / "soccer-twos-starter"
        )

    if _running_in_colab():
        try:
            from google.colab import drive  # type: ignore
            if not Path("/content/drive/MyDrive").exists():
                drive.mount("/content/drive")
        except Exception:
            pass
        for drive_root in (Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives"), Path("/content")):
            for relative in (
                Path("CS 8803- DRL") / "project" / "soccer-twos-starter",
                Path("project") / "soccer-twos-starter",
                Path("soccer-twos-starter"),
                Path("Colab Notebooks") / "soccer-twos-starter",
            ):
                yield from add(drive_root / relative)


def _find_project_root():
    for candidate in _candidate_project_roots():
        if (candidate / PROJECT_MARKER).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find soccer_twos_project/notebook_tools.py. "
        "Open this notebook from the project root/notebooks folder, or set SOCCER_TWOS_PROJECT_ROOT."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for _module_name in list(sys.modules):
    if _module_name == "soccer_twos_project" or _module_name.startswith("soccer_twos_project."):
        del sys.modules[_module_name]

importlib.invalidate_caches()
from IPython.display import Markdown, display
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

Runtime: mac
Project root: /Users/vedaangchopra/all_data/Georgia Tech/Course Content/CS 8803- DRL/project/project/soccer-twos-starter
Artifact root: /Users/vedaangchopra/all_data/Georgia Tech/Course Content/CS 8803- DRL/project/project/soccer-twos-starter/artifacts/cs8803_soccer_twos
Python: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python
soccer_twos: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/lib/python3.8/site-packages/soccer_twos/__init__.py
ray: 1.13.0
torch: 1.13.1
python: /opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python
{
  "cpu_count": 16,
  "gpu_name": "",
  "mlx_available": false,
  "ram_gb": 64.0,
  "torch_cuda_available": false,
  "torch_cuda_device_count": 0,
  "torch_cuda_device_names": [],
  "torch_cuda_runtime_probe_device": "",
  "torch_cuda_runtime_probe_error": "",
  "torch_cuda_runtime_ready": false,
  "torch_cuda_version": "",
  "torch_mps_available": true,
  "torch_version": "1.13.1"
}


## Environment Gate

Expected training setup: one `(336,)` observation vector and `Discrete(27)` actions. Expected live match setup: four player observations and `MultiDiscrete([3, 3, 3])` actions.

In [2]:
try:
    run_environment_gate(steps=10)
except Exception as exc:
    print("Environment gate failed:", type(exc).__name__, exc)
    print("Use the soccertwos conda environment before running training notebooks.")

{
  "gym": "0.19.0",
  "numpy": "1.23.5",
  "python": "/opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/bin/python",
  "ray": "1.13.0",
  "soccer_twos": "/opt/homebrew/Caskroom/miniconda/base/envs/soccertwos/lib/python3.8/site-packages/soccer_twos/__init__.py",
  "torch": "1.13.1"
}
Using Unity base_port: 50039


I0000 00:00:1777856028.971372 17591604 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


...

[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Gate reset observation shape: (336,)
First-step info: {'player_info': {'position': [-8.528812408447266, 1.2199244499206543], 'rotation_y': 87.72977447509766, 'velocity': [8.045080184936523, 0.3189372420310974]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
Environment gate passed. steps=10 total_reward=0.000


## Random Rollout Logs

These logs show the loop used by training: observation, action, reward, done, info. Rewards are sparse, so inspect `player_info` and `ball_info` when available.

In [3]:
try:
    run_random_debug_episode(max_steps=10, show_info=True)
except Exception as exc:
    print("Debug rollout skipped:", type(exc).__name__, exc)

Using Unity base_port: 50042


I0000 00:00:1777856037.618692 17591604 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Observation type: <class 'numpy.ndarray'>
Observation shape: (336,)
Action space: Discrete(27)
step=01 action=4 reward=0.0 done=False info={'player_info': {'position': [-9.018893241882324, 1.0623332262039185], 'rotation_y': 77.7297592163086, 'velocity': [0.26045092940330505, -2.1968958377838135]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=02 action=25 reward=0.0 done=False info={'player_info': {'position': [-9.033243179321289, 1.0396203994750977], 'rotation_y': 67.72975158691406, 'velocity': [-0.4148390293121338, 0.8182234764099121]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=03 action=7 reward=0.0 done=False info={'player_info': {'position': [-9.126251220703125, 1.2331129312515259], 'rotation_y': 57.729736328125, 'velocity': [-1.2962756156921387, 2.5777432918548584]}, 'ball_info': {'position': [1.090998649597168, 1.8254880905151367], 'velocity': [0.0, 0.0]}}
step=04 action=23 

## Observation And Action Tables

The model receives a vector, while our explanations decode actions into human-readable movement branches.

In [4]:
print_soccer_action_help()
display(soccer_flat_action_table().head(12))
display(soccer_named_action_table())

Live Unity action: MultiDiscrete([3, 3, 3])
Branch order: [forward_axis, right_axis, rotate_axis]
Flattened training action: Discrete(27), where index = forward*9 + right*3 + rotate
No-op is flat action 0 -> [0, 0, 0].
Random policy samples one of the 27 flat actions at every env.step().


,flat_action,branch_action,meaning
0,0,"[0, 0, 0]",no-op / stay still
1,1,"[0, 0, 1]",rotate A branch
2,2,"[0, 0, 2]",rotate D branch
3,3,"[0, 1, 0]",strafe right (E)
4,4,"[0, 1, 1]",strafe right (E) + rotate A branch
5,5,"[0, 1, 2]",strafe right (E) + rotate D branch
6,6,"[0, 2, 0]",strafe left (Q)
7,7,"[0, 2, 1]",strafe left (Q) + rotate A branch
8,8,"[0, 2, 2]",strafe left (Q) + rotate D branch
9,9,"[1, 0, 0]",forward force (W); also gives kick power on ba...


,name,flat_action,branch_action,meaning
0,noop,0,"[0, 0, 0]",no-op / stay still
1,rotate_a,1,"[0, 0, 1]",rotate A branch
2,rotate_left,1,"[0, 0, 1]",rotate A branch
3,rotate_d,2,"[0, 0, 2]",rotate D branch
4,rotate_right,2,"[0, 0, 2]",rotate D branch
5,right,3,"[0, 1, 0]",strafe right (E)
6,strafe_right,3,"[0, 1, 0]",strafe right (E)
7,left,6,"[0, 2, 0]",strafe left (Q)
8,strafe_left,6,"[0, 2, 0]",strafe left (Q)
9,forward,9,"[1, 0, 0]",forward force (W); also gives kick power on ba...


## Reward Shaping Signal

Reward shaping is training-only. The wrapper adds small bonuses for moving toward the ball and moving the ball toward goal; exported agents do not depend on this wrapper.

In [5]:
try:
    from soccer_twos import EnvType
    from soccer_twos_project.envs import RewardShapingWrapper

    render_reward_shaping_unity = resolve_unity_render_request(
        True,
        ctx=ctx,
        label="Reward shaping Unity playback",
    )
    env = RewardShapingWrapper(
        make_soccer_env(
            render=render_reward_shaping_unity,
            variation=EnvType.team_vs_policy,
            flatten_branched=True,
            single_player=True,
        ),
        player_to_ball_weight=0.01,
        ball_to_goal_weight=0.02,
        clip=0.05,
    )
    try:
        obs = env.reset()
        rows = []
        for step in range(5):
            obs, reward, done, info = env.step(env.action_space.sample())
            rows.append({"step": step, "shaped_reward": float(reward), **rollout_metrics_from_info(info)})
        display(pd.DataFrame(rows))
    finally:
        env.close()
except Exception as exc:
    print("Reward shaping demo skipped:", type(exc).__name__, exc)


Reward shaping Unity playback disabled: no live display was detected.
Using Unity base_port: 50043


I0000 00:00:1777856041.935772 17591604 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


,step,shaped_reward,player_x,player_y,ball_x,ball_y,player_ball_dist,ball_goal_dist
0,0,0.0,-9.029830,1.338225,1.090999,1.825488,10.132551,13.037435
1,1,0.0,-9.033645,1.357244,1.090999,1.825488,10.135465,13.037435
2,2,0.0,-9.040831,1.431936,1.090999,1.825488,10.139470,13.037435
3,3,0.0,-9.047911,1.551995,1.090999,1.825488,10.142597,13.037435
4,4,0.0,-9.051421,1.611530,1.090999,1.825488,10.144676,13.037435


## Key Takeaways

SoccerTwos training uses a simplified single-player view, sparse rewards, and optional training-only shaping. The final submission agent only needs to implement `AgentInterface.act()`.

## What To Run Next

Run `01_training_smoke_and_tensorboard.ipynb` to prove Ray/RLlib training, logs, and checkpoints work.